# Jobs and Workflows Examples

This notebook explains common Databricks job and workflow patterns:

- passing parameters into tasks
- using widgets for reusable notebook runs
- returning task values between workflow steps
- retries and failure handling
- scheduling with cron expressions

## Notebook parameters

Jobs usually pass runtime parameters into notebooks. Widgets are the most common pattern for that in Databricks notebooks.

In [ ]:
dbutils.widgets.text("run_date", "2026-04-04", "Run Date")
dbutils.widgets.dropdown("run_mode", "incremental", ["incremental", "full"], "Run Mode")
dbutils.widgets.text("source_path", "/Volumes/main/demo/raw/orders", "Source Path")

run_date = dbutils.widgets.get("run_date")
run_mode = dbutils.widgets.get("run_mode")
source_path = dbutils.widgets.get("source_path")

print({"run_date": run_date, "run_mode": run_mode, "source_path": source_path})

## Simple task logic

A workflow task should be deterministic and parameter-driven. This sample shows a small branching pattern.

In [ ]:
if run_mode == "full":
    selected_message = f"Running a full reload for {run_date} from {source_path}"
else:
    selected_message = f"Running an incremental load for {run_date} from {source_path}"

print(selected_message)

## Pass values between tasks

Databricks workflows allow one task to emit values that downstream tasks can consume.

In [ ]:
record_count = 1250
quality_status = "passed"

dbutils.jobs.taskValues.set(key="record_count", value=record_count)
dbutils.jobs.taskValues.set(key="quality_status", value=quality_status)

print("Published task values for downstream tasks")

In [ ]:
# Example for a downstream task notebook.
upstream_record_count = dbutils.jobs.taskValues.get(taskKey="ingest_orders", key="record_count", debugValue=0)
upstream_quality_status = dbutils.jobs.taskValues.get(taskKey="ingest_orders", key="quality_status", debugValue="unknown")

print({"record_count": upstream_record_count, "quality_status": upstream_quality_status})

## Example workflow JSON

This is a compact example of how a multi-task job can be represented.

In [ ]:
workflow_definition = {
    "name": "orders_daily_pipeline",
    "max_concurrent_runs": 1,
    "tasks": [
        {
            "task_key": "ingest_orders",
            "notebook_task": {
                "notebook_path": "/Workspace/Shared/notebooks/ingest_orders",
                "base_parameters": {
                    "run_date": "{{job.parameters.run_date}}",
                    "run_mode": "incremental"
                }
            },
            "job_cluster_key": "etl_cluster",
            "timeout_seconds": 3600
        },
        {
            "task_key": "validate_orders",
            "depends_on": [{"task_key": "ingest_orders"}],
            "notebook_task": {
                "notebook_path": "/Workspace/Shared/notebooks/validate_orders"
            },
            "job_cluster_key": "etl_cluster",
            "max_retries": 2
        },
        {
            "task_key": "publish_orders",
            "depends_on": [{"task_key": "validate_orders"}],
            "notebook_task": {
                "notebook_path": "/Workspace/Shared/notebooks/publish_orders"
            },
            "job_cluster_key": "etl_cluster"
        }
    ],
    "job_clusters": [
        {
            "job_cluster_key": "etl_cluster",
            "new_cluster": {
                "spark_version": "15.4.x-scala2.12",
                "node_type_id": "Standard_DS3_v2",
                "num_workers": 2
            }
        }
    ]
}

workflow_definition

## Scheduling

Cron expressions are commonly used for recurring jobs. Always validate the exact scheduler format used by your Databricks deployment.

In [ ]:
schedule_examples = {
    "daily_6am": "0 0 6 * * ?",
    "every_30_minutes": "0 0/30 * * * ?",
    "weekdays_8am": "0 0 8 ? * MON-FRI"
}

schedule_examples

## Practical guidance

- Keep tasks small and focused
- Prefer parameters over hardcoded dates and paths
- Use retries only for transient failures
- Use task dependencies to make execution order explicit
- Separate interactive notebook development from production jobs
- Store reusable business logic in Python modules or repos when notebooks grow too large